# Clustering Using Spherical K-means (with KeyBERT)


## Set up Environment


### Install Dependencies


In [1]:
!pip install --upgrade numpy pandas scikit-learn tqdm


In [2]:
!pip install --upgrade faiss-cpu


In [3]:
!pip install --upgrade keybert


### Import Dependencies


In [4]:
import os
import csv
import faiss
import numpy as np
import pandas as pd

from keybert import KeyBERT
from sklearn.metrics import silhouette_score
from tqdm import tqdm


c:\Users\Najib\miniconda3\envs\ta_510_taxodev\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load Data


In [5]:
df = pd.read_pickle('datasets/tickets.pkl')

X_embeddings = np.vstack(df['EMBEDDING'].values)
embedding_dimension = X_embeddings.shape[1]

print("Vektor embedding berhasil dimuat kembali.")
print(f"Jumlah data: {X_embeddings.shape[0]}")
print(f"Dimensi embedding: {embedding_dimension}")

df.head()


Vektor embedding berhasil dimuat kembali.
Jumlah data: 1639
Dimensi embedding: 256


,RAW TICKET,CLEANED TICKET,EMBEDDING,WORDCLOUD TICKET
0,Klien: Setwapres Medsos\nisu: Crawlback Commen...,crawlback comment. sapaan tim it sapaan bantua...,"[0.00083771703, -0.58585614, 0.064690515, 0.08...",crawlback comment. bantuan crawlback comment m...
1,"selamat pagi tim it, mohon bantuannya saya men...","sapaan tim it, sapaan saya menemukan link dari...","[0.02750907, 0.037827827, -0.5283966, -0.13133...",menemukan megapolitan.kompas.com tidak masuk t...
2,Klien: BPS\nIsu: Data postingan Instagram tida...,data postingan instagram tidak masuk. sapaan t...,"[-0.17697816, -0.045605347, -0.3471804, 0.4249...",instagram tidak masuk. apa bikin instagram lam...
3,selamat sore mas @Dhanysybn dan tim info untuk...,sapaan mas nama orang dan tim info untuk siput...,"[0.08603829, -0.017807081, 0.016980363, 0.3204...",siputri tidak bisa akses minta yaaa.
4,Klien: Heinz \nIsu: Dashboard loading\n\nSelam...,"dashboard loading. sapaan tim it, sapaan ada k...","[0.14301413, 0.028007613, 0.030197145, 0.24381...",loading. keluhan pihak heinz heinz loading.


## Experimenting


### Normalize Data


In [6]:
X_embeddings = np.ascontiguousarray(X_embeddings, dtype=np.float32)
faiss.normalize_L2(X_embeddings)

print("Vektor embedding berhasil dinormalisasi.")


Vektor embedding berhasil dinormalisasi.


### Initialize KeyBERT


In [7]:
# Menggunakan model yang sama dengan IndoSBERT agar ruang semantik konsisten.
# Ganti dengan path/nama model IndoSBERT yang dipakai di data_preparation.ipynb jika berbeda.
INDOSBERT_MODEL = 'denaya/indoSBERT-large'  # sesuaikan jika berbeda

kw_model = KeyBERT(model=INDOSBERT_MODEL)
print(f"KeyBERT berhasil diinisialisasi dengan model: {INDOSBERT_MODEL}")


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 13713.18it/s]


KeyBERT berhasil diinisialisasi dengan model: denaya/indoSBERT-large


### Initialize Spherical K-means


In [8]:
def run_skm(dimension, ncluster, X, niter=20):
    skm = faiss.Kmeans(
        d=dimension,
        k=ncluster,
        niter=niter,
        verbose=True,
        spherical=True
    )
    skm.train(X)

    centroids = skm.centroids
    _, labels = skm.index.search(X, 1)

    return centroids, labels.flatten()


### KeyBERT Keyword Extraction per Cluster


In [9]:
def extract_keywords_keybert(
    texts: list,
    top_n: int = 5,
    keyphrase_ngram_range: tuple = (1, 2),
    use_mmr: bool = True,
    diversity: float = 0.5
) -> str:
    """
    Menggabungkan semua teks dalam satu klaster menjadi satu dokumen,
    lalu mengekstrak keyword representatif menggunakan KeyBERT.

    Mengapa teks digabung (concatenate) bukan dirata-rata embedding-nya?
    KeyBERT bekerja pada level teks: ia membandingkan embedding kandidat
    frasa dengan embedding dokumen keseluruhan. Menggabungkan teks memberi
    distribusi kata yang merepresentasikan seluruh klaster, sehingga
    keyword yang terpilih bersifat kolektif—bukan hanya mewakili satu tiket.

    Args:
        texts                : list teks milik klaster (dari kolom CLEANED TICKET).
        top_n                : jumlah keyword/keyphrase yang diambil.
        keyphrase_ngram_range: rentang n-gram; (1,1) = unigram; (1,2) = uni+bigram.
        use_mmr              : gunakan Maximal Marginal Relevance untuk diversifikasi;
                               mencegah keyword yang semantically redundant.
        diversity            : tingkat keragaman MMR [0.0–1.0]; 0.5 = keseimbangan
                               relevansi vs diversitas.

    Returns:
        String berformat "(keyword, score%); ..." yang siap disimpan ke CSV.
        Mengembalikan string kosong jika teks tidak tersedia.
    """
    clean_texts = [t for t in texts if isinstance(t, str) and t.strip()]
    if not clean_texts:
        return ""

    # Gabung seluruh teks klaster menjadi satu dokumen representatif
    combined_doc = " ".join(clean_texts)

    keywords = kw_model.extract_keywords(
        combined_doc,
        keyphrase_ngram_range=keyphrase_ngram_range,
        stop_words=None,      # stopword removal tidak diperlukan karena IndoSBERT
                              # sudah menghasilkan representasi kontekstual
        use_mmr=use_mmr,
        diversity=diversity,
        top_n=top_n
    )

    # Format: (keyword, skor_cosine_similarity_persen%)
    return "; ".join(f"({kw}, {score*100:.2f}%)" for kw, score in keywords)


### Cosine Similarity Intra-Cluster


In [10]:
def cosine_similarity_intra_cluster(embeddings, labels, centroids):
    centroids_norm = centroids / np.linalg.norm(centroids, axis=1, keepdims=True)
    sims = np.sum(embeddings * centroids_norm[labels], axis=1)
    return float(np.mean(sims))


### Run Experiments Scenario


In [ ]:
min_clusters = 2
max_clusters = 10
k_range = range(min_clusters, max_clusters + 1)

results = []
scenario_count = 0

# KeyBERT membutuhkan teks yang sudah bersih namun masih natural-language
# (bukan token list). Kolom CLEANED TICKET adalah pilihan yang tepat.
keybert_texts = df['CLEANED TICKET'].astype(str).values

total_iterations = sum(k1 * len(k_range) for k1 in k_range)

with tqdm(total=total_iterations, desc="Evaluasi + KeyBERT") as pbar:

    for k1 in k_range:
        centroids_l1, labels_l1 = run_skm(embedding_dimension, k1, X_embeddings)
        silhouette_avg_l1 = silhouette_score(X_embeddings, labels_l1, metric='cosine')

        n_output_l1 = len(X_embeddings)
        n_clustered_l1 = np.sum(labels_l1 >= 0)

        # --- Pra-hitung keyword L1 & cache data klaster ---
        # Keyword L1 dihitung SEKALI per (k1, cluster_id) agar tidak diulang
        # untuk setiap nilai k2 — menghemat biaya komputasi KeyBERT secara signifikan.
        cluster_cache_l1 = {}
        keywords_l1_cache = {}

        for cluster_id in range(k1):
            cluster_indices = np.where(labels_l1 == cluster_id)[0]
            cluster_embeddings = X_embeddings[cluster_indices]
            cluster_texts = keybert_texts[cluster_indices].tolist()

            cluster_cache_l1[cluster_id] = (cluster_embeddings, cluster_texts, cluster_indices)
            keywords_l1_cache[cluster_id] = extract_keywords_keybert(cluster_texts)

        for cluster_id in range(k1):
            cluster_embeddings, cluster_texts, cluster_indices = cluster_cache_l1[cluster_id]
            top_words_l1 = keywords_l1_cache[cluster_id]

            for k2 in k_range:
                if len(cluster_embeddings) >= k2:
                    centroids_l2, labels_l2 = run_skm(embedding_dimension, k2, cluster_embeddings)
                    silhouette_avg_l2 = silhouette_score(cluster_embeddings, labels_l2, metric='cosine')

                    n_output_l2 = len(cluster_embeddings)
                    n_clustered_l2 = np.sum(labels_l2 >= 0)

                    # --- KeyBERT Level 2: per sub-klaster ---
                    l2_keywords_parts = []
                    for sub_id in range(k2):
                        sub_indices = np.where(labels_l2 == sub_id)[0]
                        sub_texts = [cluster_texts[i] for i in sub_indices]
                        sub_keywords = extract_keywords_keybert(sub_texts)
                        l2_keywords_parts.append(
                            f"(sub-cluster id, {sub_id + 1}; [{sub_keywords}])"
                        )

                    top_words_l2 = "; ".join(l2_keywords_parts)

                    # --- Simpan Pemetaan Ticket -> Cluster & Sub-cluster ---
                    sub_cluster_ids = labels_l2 + 1

                    labels_df = pd.DataFrame({
                        'ticket': [df['RAW TICKET'].iloc[idx] for idx in cluster_indices],
                        'cluster_id': cluster_id + 1,
                        'sub_cluster_id': sub_cluster_ids
                    })

                    labels_dir = f"results/labels/k{k1}/cluster_{cluster_id+1}_k{k2}"
                    os.makedirs(labels_dir, exist_ok=True)
                    labels_df.to_csv(
                        f"{labels_dir}/labels.csv",
                        index=False,
                        encoding='utf-8',
                        quoting=csv.QUOTE_ALL
                    )

                    # --- Ending Conditions ---

                    # Objective Condition 1: semua input L1 dan L2 berhasil diklastering
                    objective_condition_1 = "YES" if (
                        n_clustered_l1 == n_output_l1 and n_clustered_l2 == n_output_l2
                    ) else "NO"

                    # Subjective Concise: k1 dan k2 berada dalam rentang [2, 10]
                    subjective_concise = "YES" if (2 <= k1 <= 10 and 2 <= k2 <= 10) else "NO"

                    # Subjective Robust: silhouette dalam [0, 0.5] DAN cosine intra >= threshold
                    cosine_sim_intra_l1 = cosine_similarity_intra_cluster(
                        X_embeddings, labels_l1, centroids_l1
                    )
                    cosine_sim_intra_l2 = cosine_similarity_intra_cluster(
                        cluster_embeddings, labels_l2, centroids_l2
                    )
                    COSINE_SIM_THRESHOLD = 0.75
                    is_silhouette_in_range = (
                        (0 <= silhouette_avg_l1 <= 0.5) and (0 <= silhouette_avg_l2 <= 0.5)
                    )
                    is_cosine_near_one = (
                        (cosine_sim_intra_l1 >= COSINE_SIM_THRESHOLD)
                        and (cosine_sim_intra_l2 >= COSINE_SIM_THRESHOLD)
                    )
                    subjective_robust = "YES" if (is_silhouette_in_range and is_cosine_near_one) else "NO"

                    # Subjective Comprehensive: semua tiket di L2 berhasil diklastering
                    subjective_comprehensive = "YES" if n_clustered_l2 == n_output_l2 else "NO"

                    results.append({
                        'scenario_id': f"{k1-1}.{cluster_id+1}.{k2-1}",
                        'cluster_id': f"{k1}.{cluster_id+1}",
                        'k1': k1,
                        'k2': k2,
                        'silhouette_avg_l1': f"{silhouette_avg_l1:.4f}",
                        'silhouette_avg_l2': f"{silhouette_avg_l2:.4f}",
                        'cosine_sim_intra_l1': f"{cosine_sim_intra_l1:.4f}",
                        'cosine_sim_intra_l2': f"{cosine_sim_intra_l2:.4f}",
                        'n_output_l1': n_output_l1,
                        'n_clustered_l1': n_clustered_l1,
                        'n_output_l2': n_output_l2,
                        'n_clustered_l2': n_clustered_l2,
                        'objective_condition_1': objective_condition_1,
                        'subjective_concise': subjective_concise,
                        'subjective_robust': subjective_robust,
                        'subjective_comprehensive': subjective_comprehensive,
                        'top_words_l1': top_words_l1,
                        'top_words_l2': top_words_l2
                    })
                    scenario_count += 1

                pbar.update(1)

print(f"\nTotal skenario yang berhasil dievaluasi: {scenario_count}")
print("Keyword tiap klaster berhasil diekstrak dengan KeyBERT.")


Evaluasi + KeyBERT:   0%|          | 0/486 [00:00<?, ?it/s]

### Ending Condition: Subjective Extendible


In [ ]:
results_df_temp = pd.DataFrame(results)
results_df_temp['silhouette_avg_l1'] = results_df_temp['silhouette_avg_l1'].astype(float)
results_df_temp['silhouette_avg_l2'] = results_df_temp['silhouette_avg_l2'].astype(float)

best_sil_l1 = results_df_temp['silhouette_avg_l1'].max()
best_sil_l2_per_cluster = results_df_temp.groupby('cluster_id')['silhouette_avg_l2'].max()

for r in results:
    sil_l1_val = float(r['silhouette_avg_l1'])
    sil_l2_val = float(r['silhouette_avg_l2'])
    cluster_id_val = r['cluster_id']

    is_best_l1 = (sil_l1_val == best_sil_l1)
    is_best_l2 = (sil_l2_val == best_sil_l2_per_cluster[cluster_id_val])

    r['subjective_extendible'] = "YES" if (is_best_l1 and is_best_l2) else "NO"

print("Kolom 'subjective_extendible' berhasil ditambahkan ke seluruh hasil skenario.")


### Save Results


In [ ]:
file_path = 'results/experiments/results.csv'
os.makedirs(os.path.dirname(file_path), exist_ok=True)

if results:
    fieldnames = list(results[0].keys())

    with open(file_path, 'w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames, quoting=csv.QUOTE_ALL)
        writer.writeheader()
        writer.writerows(results)

    print(f"Results saved to {file_path}")
